In [2]:
import openai
import re
import time
import json

import numpy as np

from tqdm import tqdm
from pprint import pprint
from tenacity import retry, stop_after_attempt, wait_chain, wait_fixed

import os
from openai import AzureOpenAI

import math

import re
import math
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import traceback

In [9]:
endpoint = "https://pankajaiml.openai.azure.com/"
model_name = "gpt-4o"
deployment = "gpt-4o"
subscription_key = "REDACTED_AZURE_OPENAI_KEY"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Retry logic
@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=1512,
        temperature=0.0,
        model=deployment
    )

In [4]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as reader:
        data = json.load(reader)  # Load the entire JSON file
    return data

dev_data = load_json('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/AddSubsampled_train.json')
CoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/CoT_prompt_examples.txt').read()
Standard_prompt_examples = open("/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/standard_prompt_examples.txt").read()
CCoT_prompt_examples = open("/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/prompt_examples/CCoT_prompt_example.txt").read()

In [10]:
import concurrent.futures
import math
import re
from tqdm import tqdm

# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/outputs/CoT_prompt_1.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        return num
    except ValueError:
        return None

# === Processing Function ===
def process_data(d):
    global acc, total
    q = d['question']
    a = float(d['correct'][0])  # Ground truth

    prompt_q = (
        CoT_prompt_examples +
        '\nQ: ' + q + " Think step by step. Write your answer as: the answer is <answer>"
    )

    messages = [
        {"role": "system", "content": "Your goal is to answer these math questions accurately."},
        {"role": "user", "content": prompt_q}
    ]

    response = completion_with_backoff(messages)

    # === Validate Response ===
    if response and response.choices and response.choices[0].message and response.choices[0].message.content:
        ans_model = response.choices[0].message.content.strip()
    else:
        return f"❌ Invalid response\nQ: {q}\nResponse: {response}\n\n", False

    # === Extract Answer
        match = re.search(
            r'(?:the answer is|final answer:)\s*\**\$?(-?\d+(?:\.\d+)?)\**\s*(?:[a-zA-Z%$ ]+)?[\.]?',
            ans_model,
            re.IGNORECASE
        )
        if match:
            extracted_raw = match.group(1).strip()
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

    # === Log Block
    log_block = (
        f'Q: {q}\n'
        f'A_model:\n{ans_model}\n'
        f'Extracted:\n{extracted}\n'
        f'A:\n{a}\n\n'
    )

    # === Write to Correct/Incorrect Logs
    if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
        acc += 1
        return log_block, True
    else:
        return "❌ Incorrect or Invalid\n" + log_block, False

# === Main Loop with Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_data, d) for d in dev_data]
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(dev_data)):
            log_block, is_correct = future.result()
            if is_correct:
                fd.write(log_block)
            else:
                bad_fd.write(log_block)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 0/200 [00:14<?, ?it/s]


In [6]:
# === Metrics ===
acc = 0
total = 0

# === File Output Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/outputs/Standard_prompt_1.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)  # Keep digits, decimal, minus
    try:
        num = float(cleaned)
        return round(num, 4)
    except ValueError:
        return None

# === Main Loop ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    for d in tqdm(dev_data):
        q = d['question']
        a = float(d['correct'][0])  # Ground truth

        prompt_q = (
            Standard_prompt_examples +
            '\nAnswer this question: ' + q + " Write your answer as: the answer is <answer>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions accuractly"},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Improved Answer Extraction ===
        match = re.search(r'the answer is\s*([-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            acc += 1
            fd.write(log_block)
        else:
            bad_fd.write("❌ Incorrect or Invalid\n" + log_block)

        total += 1
        print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 1/200 [00:00<02:39,  1.25it/s]

Accuracy: 1 / 1 = 100.00%


  1%|          | 2/200 [00:01<02:58,  1.11it/s]

Accuracy: 1 / 2 = 50.00%


  2%|▏         | 3/200 [00:02<02:36,  1.26it/s]

...

Accuracy: 189 / 198 = 95.45%


100%|█████████▉| 199/200 [05:13<00:01,  1.35s/it]

Accuracy: 190 / 199 = 95.48%


100%|██████████| 200/200 [05:14<00:00,  1.57s/it]

Accuracy: 191 / 200 = 95.50%


In [10]:
# === Metrics ===
acc = 0
total = 0

# === File Output Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/GPT4_Turbo/outputs/CCoT_prompt.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        return round(num, 4)
    except ValueError:
        return None

# === Main Loop ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    for d in tqdm(dev_data):
        q = d['question']
        a = float(d['correct'][0])  # Ground truth answer

        # === Prompt Setup for Complex CoT ===
        prompt_q = (
            CCoT_prompt_examples +
            "\nQ: " + q + "\n\n"
            "Please reason through this problem using a complex, multi-step chain of thought:\n"
            "Step 1: Clearly state all given information and any assumptions.\n"
            "Step 2: Propose two different methods to solve the problem, briefly outlining the logic of each.\n"
            "Step 3: For each method, work through all intermediate steps in detail, showing calculations, checks, and potential pitfalls.\n"
            "Step 4: Evaluate and compare the two methods—discussing which is better based on clarity, reliability, or efficiency.\n"
            "Step 5: Choose the better method and use it to solve the problem, showing all steps.\n"
            "Step 6: Double-check the solution for errors or unreasonable results.\n"
            "Finish your response with: the answer is <answer>"
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "Your goal is to answer the question using a complex, coherent, step by step thoughts, answering the questions correctly.\n"
                )
            },
            {"role": "user", "content": prompt_q}
        ]

        # === Get Response ===
        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Improved Answer Extraction ===
        match = re.search(
            r'(?:the answer is|final answer:)\s*\**\$?(-?\d+(?:\.\d+)?)\**\s*(?:[a-zA-Z%$ ]+)?[\.]?',
            ans_model,
            re.IGNORECASE
        )
        if match:
            extracted_raw = match.group(1).strip()
            extracted = clean_and_truncate(extracted_raw)
        else:
            extracted = None

        # === Log Block
        log_block = (
            f'Q: {q}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted:\n{extracted}\n'
            f'A:\n{a}\n\n'
        )

        if extracted is not None and math.isclose(extracted, a, rel_tol=1e-4):
            acc += 1
            fd.write(log_block)
        else:
            bad_fd.write("❌ Incorrect or Invalid\n" + log_block)

        total += 1
        print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

  0%|          | 1/200 [00:06<20:55,  6.31s/it]

Accuracy: 1 / 1 = 100.00%


  1%|          | 2/200 [00:20<36:36, 11.09s/it]

Accuracy: 1 / 2 = 50.00%


  2%|▏         | 3/200 [00:28<31:46,  9.68s/it]

...

Accuracy: 173 / 198 = 87.37%


100%|█████████▉| 199/200 [29:03<00:10, 10.03s/it]

Accuracy: 174 / 199 = 87.44%


100%|██████████| 200/200 [29:15<00:00,  8.78s/it]

Accuracy: 175 / 200 = 87.50%
